# 02. 데이터 전처리

웨이퍼 맵을 고정 크기 이미지 텐서로 변환해 학습 속도를 올립니다.

**입력:** `data/LSWMD.pkl` + `data/processed/labeled_index.parquet`
**출력:** `data/processed/wafer_{train,val,test}.npz` (X, y 배열)


## 1. 설정 로드

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import cv2
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

IMG_SIZE = cfg['data']['image_size']
CLASSES = cfg['data']['classes']
VAL_RATIO = cfg['data']['val_ratio']
TEST_RATIO = cfg['data']['test_ratio']
SEED = cfg['project']['seed']

label2idx = {c: i for i, c in enumerate(CLASSES)}
print(f"image_size={IMG_SIZE}, classes={CLASSES}")


image_size=128, classes=['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']


## 2. 원본 + 인덱스 로드

In [2]:
df = pd.read_pickle('../data/LSWMD.pkl')
idx = pd.read_parquet('../data/processed/labeled_index.parquet')
print(f"원본: {df.shape}, 라벨 인덱스: {idx.shape}")

# 학습 대상만 추출
df_sub = df.loc[idx['index']].copy()
df_sub['label'] = idx['label'].values
df_sub = df_sub[df_sub['label'].isin(CLASSES)].reset_index(drop=True)
print(f"대상: {len(df_sub):,}장")
print(df_sub['label'].value_counts())


원본: (811457, 6), 라벨 인덱스: (172930, 6)
대상: 172,930장
label
none         147425
Edge-Ring      9677
Edge-Loc       5189
Center         4292
Loc            3588
Scratch        1190
Random          865
Donut           555
Near-full       149
Name: count, dtype: int64


## 3. 리사이즈 함수

웨이퍼 맵 값: 0(미검사)/1(정상)/2(불량)  
RGB 3채널로 인코딩 (또는 1채널 유지 가능).

In [3]:
def wafer_to_image(wmap: np.ndarray, size: int = 128) -> np.ndarray:
    """웨이퍼 맵을 size x size uint8 이미지로 변환."""
    # 최근접 보간 (원값 보존)
    resized = cv2.resize(wmap.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    # 1채널 유지 (모델 input은 필요 시 3채널 복제)
    return resized  # (H, W), values in {0, 1, 2}

# 샘플 확인
sample = df_sub.iloc[0]['waferMap']
img = wafer_to_image(sample, IMG_SIZE)
print(f"원본 {sample.shape} -> {img.shape}, unique values: {np.unique(img)}")


원본 (45, 48) -> (128, 128), unique values: [0 1 2]


## 4. 전체 변환 및 분할

In [4]:
X = np.zeros((len(df_sub), IMG_SIZE, IMG_SIZE), dtype=np.uint8)
y = np.zeros(len(df_sub), dtype=np.int64)

for i, row in enumerate(tqdm(df_sub.itertuples(index=False), total=len(df_sub))):
    X[i] = wafer_to_image(row.waferMap, IMG_SIZE)
    y[i] = label2idx[row.label]

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"메모리: {X.nbytes/1e6:.1f} MB")


  0%|          | 0/172930 [00:00<?, ?it/s]

X shape: (172930, 128, 128), y shape: (172930,)
메모리: 2833.3 MB


In [5]:
# train/val/test stratified 분할
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, stratify=y, random_state=SEED
)
val_ratio_adj = VAL_RATIO / (1 - TEST_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_ratio_adj, stratify=y_trainval, random_state=SEED
)
print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")


train: (121050, 128, 128), val: (25940, 128, 128), test: (25940, 128, 128)


## 5. 저장

In [6]:
out_dir = Path('../data/processed')
np.savez_compressed(out_dir / 'wafer_train.npz', X=X_train, y=y_train)
np.savez_compressed(out_dir / 'wafer_val.npz',   X=X_val,   y=y_val)
np.savez_compressed(out_dir / 'wafer_test.npz',  X=X_test,  y=y_test)
print('저장 완료')
for f in sorted(out_dir.glob('wafer_*.npz')):
    print(f'  {f.name}: {f.stat().st_size/1e6:.1f} MB')


저장 완료
  wafer_test.npz: 8.0 MB
  wafer_train.npz: 37.4 MB
  wafer_val.npz: 8.0 MB


## 다음 단계

- `03_baseline_model.ipynb`: ResNet-18/EfficientNet-B0 파인튜닝
